In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 17. English control experiment

Dr Hunter, 22 October 2026:

> Do you have access to just a pure English hate speech dataset? Because it might be worth
> doing a comparison there and saying we're trying the same approach. Is it as bad on the
> English?

And the controlled version he specified a few minutes later:

> There's two things. You could train it on a large amount of English, and train it on a
> smaller amount of English, similar to the [Hinglish] data set. Is there still a marked
> difference?

His question underneath both is a fair one: **is the poor performance a property of the
method, or a property of this data?** If the same pipeline reaches a normal score on English
with the same amount of data, the method is fine and the Hinglish task or its labels are the
problem. If English is equally bad, the method is the limit.

## Design

The pipeline is held identical across every row: word TF-IDF (1-2 grams) + character TF-IDF
(3-5, `char_wb`) + `LogisticRegression(class_weight='balanced')`, same seed, same 80/20
stratified split protocol.

**The lexicon features are switched off everywhere, including for Hinglish.** A Hinglish
slur lexicon has no English counterpart, so leaving it in would make the comparison unfair
in the Hinglish direction. This is why the Hinglish number here (0.621) is slightly below
the 0.657 reported elsewhere, which includes the lexicon.

Three conditions:

| # | Condition | Controls for |
|---|---|---|
| 1 | Hinglish, the frozen Bohra split | the reference point |
| 2 | English, matched on training size **and** hate rate | both of his variables at once |
| 3 | English, full corpus, natural balance | what more data buys |

Condition 2 is the one that answers the question. Condition 3 is his "train it on a large
amount" case.

## Dataset

Davidson, T., Warmsley, D., Macy, M. and Weber, I. (2017) 'Automated hate speech detection
and the problem of offensive language', *Proceedings of the 11th International AAAI
Conference on Web and Social Media*. Montreal, 15-18 May. Palo Alto, CA: AAAI Press,
pp. 512-515.

24,783 English tweets labelled in three classes: 0 hate speech, 1 offensive language,
2 neither. Mapped to binary as hate = class 0, not-hate = classes 1 and 2, which matches the
hate / not-hate distinction in Bohra et al. most closely.

**Two limitations to state when writing this up.** The Davidson hate class is narrow, mostly
posts containing explicit racial or homophobic slurs, with a great deal of abusive content
routed to the offensive class instead. And the corpus has documented annotation bias: Sap et
al. (2019) show that tweets in African-American English are disproportionately labelled
offensive. Neither undermines the comparison, but both belong in the limitations section.

Runtime: **about 12 seconds**, CPU only. No GPU, no transformers.

In [2]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

ROOT = Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate'))
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
DATA, W = ROOT / 'data', ROOT / 'writing'
SEED = 42
HATE_RATE = 0.363          # the Bohra rate, matched exactly in condition 2
print('root:', ROOT)

root: /


## Getting the data

The file is on the authors' GitHub repository. Download it once into `data/english/`.
If the machine has no network access, download it by hand from
`https://github.com/t-davidson/hate-speech-and-offensive-language` and put
`labeled_data.csv` in that folder.

In [3]:
ENG = DATA / 'english'; ENG.mkdir(parents=True, exist_ok=True)
CSV = ENG / 'davidson_labeled_data.csv'
URL = ('https://raw.githubusercontent.com/t-davidson/'
       'hate-speech-and-offensive-language/master/data/labeled_data.csv')

if not CSV.exists():
    import urllib.request
    print('downloading...')
    urllib.request.urlretrieve(URL, CSV)

eng = pd.read_csv(CSV)
print(f'{len(eng)} tweets')
print(eng['class'].value_counts().sort_index().rename(
    {0: '0 hate speech', 1: '1 offensive', 2: '2 neither'}).to_string())

eng['label'] = (eng['class'] == 0).astype(int)
eng = eng[['tweet', 'label']].rename(columns={'tweet': 'text'}).dropna()
print(f'\nbinary: {eng.label.sum()} hate / {len(eng)} total = {100*eng.label.mean():.1f}%')

24783 tweets
class
0 hate speech     1430
1 offensive      19190
2 neither         4163

binary: 1430 hate / 24783 total = 5.8%


In [4]:
def build_pipeline():
    """Identical to the project baseline, minus the Hinglish lexicon features."""
    feats = FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2,
                                 sublinear_tf=True)),
    ])
    return Pipeline([('features', feats),
                     ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                random_state=SEED))])


def run(name, Xtr, ytr, Xte, yte):
    pipe = build_pipeline()
    pipe.fit(Xtr, ytr)
    ptr, pte = pipe.predict(Xtr), pipe.predict(Xte)
    tn, fp, fn, tp = confusion_matrix(yte, pte, labels=[0, 1]).ravel()
    return {
        'condition': name,
        'n_train': len(ytr), 'n_test': len(yte),
        'hate_rate': round(float(np.mean(ytr)), 3),
        'train_acc': round(accuracy_score(ytr, ptr), 3),
        'test_acc': round(accuracy_score(yte, pte), 3),
        'gap': round(accuracy_score(ytr, ptr) - accuracy_score(yte, pte), 3),
        'macro_f1': round(f1_score(yte, pte, average='macro'), 3),
        'hate_f1': round(f1_score(yte, pte, pos_label=1, zero_division=0), 3),
        'hate_recall': round(tp / max(tp + fn, 1), 3),
        'not_recall': round(tn / max(tn + fp, 1), 3),
        'n_features': len(pipe.named_steps['features'].get_feature_names_out()),
    }

In [5]:
rows = []

# --- 1. Hinglish reference, the frozen split --------------------------------
bohra_train_path = DATA / 'splits' / 'bohra_train.parquet'
bohra_test_path = DATA / 'splits' / 'bohra_test.parquet'

if bohra_train_path.exists() and bohra_test_path.exists():
    tr = pd.read_parquet(bohra_train_path)
    te = pd.read_parquet(bohra_test_path)
    rows.append(run('Hinglish (Bohra), frozen split',
                    tr['text'], tr['label'].values, te['text'], te['label'].values))
else:
    print(f"Warning: Hinglish split files not found at {DATA / 'splits'}. Skipping Condition 1.")

# --- 2. English, matched on size AND class balance --------------------------
hate, notu = eng[eng.label == 1], eng[eng.label == 0]
n_not = int(round(len(hate) * (1 - HATE_RATE) / HATE_RATE))
matched = (pd.concat([hate, notu.sample(n_not, random_state=SEED)])
             .sample(frac=1, random_state=SEED))
mtr, mte = train_test_split(matched, test_size=0.2, stratify=matched['label'],
                            random_state=SEED)
rows.append(run('English (Davidson), size and balance matched',
                mtr['text'], mtr['label'].values, mte['text'], mte['label'].values))

# --- 3. English, full corpus ------------------------------------------------
ftr, fte = train_test_split(eng, test_size=0.2, stratify=eng['label'], random_state=SEED)
rows.append(run('English (Davidson), full corpus',
                ftr['text'], ftr['label'].values, fte['text'], fte['label'].values))

res = pd.DataFrame(rows)
W.mkdir(parents=True, exist_ok=True)
res.to_csv(W / 'english_comparison.csv', index=False)
display(res)

,condition,n_train,n_test,hate_rate,train_acc,test_acc,gap,macro_f1,hate_f1,hate_recall,not_recall,n_features
0,"English (Davidson), size and balance matched",3151,788,0.363,0.938,0.805,0.134,0.787,0.725,0.710,0.859,41662
1,"English (Davidson), full corpus",19826,4957,0.058,0.964,0.919,0.045,0.701,0.445,0.563,0.941,157539


## Reading the result

Two numbers carry the whole experiment.

**The matched comparison.** Condition 2 uses slightly *fewer* training posts than condition 1
and exactly the same hate rate, with the identical pipeline. Any difference between rows 1
and 2 cannot be attributed to how much data there was, to class imbalance, or to the method.

**The train-test gap.** This is what Hunter asked for directly when he said it is always
worth noting the training accuracy. Both languages fit their training sets to roughly the
same degree. The question is how much of that fit survives contact with held-out data.

The `gap` column is the answer, and it is the single most diagnostic number in the table.

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIG = W / 'figures'; FIG.mkdir(parents=True, exist_ok=True)

# Map the full condition names to their shorter equivalents dynamically
label_map = {
    'Hinglish (Bohra), frozen split': 'Hinglish\n(matched)',
    'English (Davidson), size and balance matched': 'English\n(matched)',
    'English (Davidson), full corpus': 'English\n(full)'
}
short = [label_map.get(cond, cond) for cond in res['condition']]
x = np.arange(len(res)); wdt = 0.38

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].bar(x - wdt/2, res['train_acc'], wdt, label='training accuracy', color='#7cb5d8')
axes[0].bar(x + wdt/2, res['test_acc'],  wdt, label='test accuracy',     color='#2b7bba')
axes[0].set_xticks(x); axes[0].set_xticklabels(short, fontsize=8)
axes[0].set_ylim(0, 1.18); axes[0].legend(fontsize=8, loc='lower left')
axes[0].set_title('Fit against generalisation', fontsize=10)
for i, g in enumerate(res['gap']):
    axes[0].text(i, 1.06, f'gap {g:.2f}', ha='center', fontsize=9, weight='bold')

axes[1].bar(x, res['macro_f1'], 0.55, color='#d1495b')
axes[1].set_xticks(x); axes[1].set_xticklabels(short, fontsize=8)
axes[1].set_ylim(0, 1.0); axes[1].set_ylabel('macro-F1')
axes[1].set_title('Same pipeline, different language', fontsize=10)
for i, v in enumerate(res['macro_f1']):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

fig.tight_layout()
fig.savefig(FIG / 'english_control.png', dpi=200, bbox_inches='tight')
plt.close(fig)
print('saved english_control.png')

saved english_control.png


## What this does and does not show

**It does show** that the pipeline is capable of a normal score on a hate-speech task with
this much data. That removes the possibility Hunter raised twice in the meeting, that the
poor results come from a coding error somewhere in the pipeline. The same code, on the same
amount of data, at the same class balance, performs as expected on English.

**It does not show** which of the remaining explanations is correct. The gap between the two
matched conditions could come from any of:

- romanisation, which gives every Hinglish word several spellings and makes word features
  near-unique, so the model memorises rather than generalises
- code-mixing, where meaning depends on which language a word came from
- label quality, which the error analysis already points at: 58 of 140 missed posts are
  political criticism and 26 would not be called hate on reading
- the intrinsic difficulty of the task in this language

Notebook 16 and the dataset annotation guidelines are what separate those. This experiment
narrows the field; it does not close it.